# 13 — Dispersive Shift Measurement

Legacy experiment 30.

Measure the dispersive shift χ = f_resonator(|g⟩) − f_resonator(|e⟩) by performing
resonator spectroscopy with and without a qubit π-pulse (x180) excitation.

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    ResonatorSpectroscopyX180,
    load_stage_checkpoint,
    open_notebook_stage,
    preview_or_apply_patch_ops,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="13_dispersive_shift_measurement",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

coherence_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="06_coherence_experiments",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"Closed a live in-memory session before reopen: {stage.had_live_session}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
print(f"Current readout frequency: {float(attr.ro_fq) / 1e9:.6f} GHz")
if coherence_checkpoint is not None:
    print(
        "Prior stage 06 status: "
        f"{coherence_checkpoint['status']}"
        f" ({coherence_checkpoint['summary']})"
    )

## 2. Dispersive Shift Defaults

In [ ]:
APPLY_CHI_CALIBRATION = True

CHI_FREQ_START = 8592 * u.MHz
CHI_FREQ_END = 8598.5 * u.MHz
CHI_DF = 50 * u.kHz
CHI_N_AVG = 20_000

print("Dispersive shift measurement settings:")
print(f"  apply χ calibration: {APPLY_CHI_CALIBRATION}")
print(f"  frequency range: {CHI_FREQ_START / 1e9:.6f} – {CHI_FREQ_END / 1e9:.6f} GHz")
print(f"  df: {CHI_DF / 1e3:.1f} kHz")
print(f"  n_avg: {CHI_N_AVG}")

## 3. Resonator Spectroscopy — Ground State (|g⟩)

Run resonator spectroscopy without qubit excitation to measure f_resonator(|g⟩).

In [ ]:
chi_exp = ResonatorSpectroscopyX180(session)

# Ground state: r180="zero" (no excitation pulse)
result_g = chi_exp.run(
    freq_start=CHI_FREQ_START,
    freq_end=CHI_FREQ_END,
    df=CHI_DF,
    r180="zero",
    n_avg=CHI_N_AVG,
)
analysis_g = chi_exp.analyze(result_g, update_calibration=False)
chi_exp.plot(analysis_g)

f_g_hz = float(analysis_g.metrics.get("f0", np.nan))
print(f"Resonator frequency (|g⟩): {f_g_hz / 1e9:.6f} GHz")

## 4. Resonator Spectroscopy — Excited State (|e⟩)

Run resonator spectroscopy with a qubit x180 pre-pulse to measure f_resonator(|e⟩).

In [ ]:
# Excited state: default r180 (x180 pre-pulse)
result_e = chi_exp.run(
    freq_start=CHI_FREQ_START,
    freq_end=CHI_FREQ_END,
    df=CHI_DF,
    n_avg=CHI_N_AVG,
)
analysis_e = chi_exp.analyze(result_e, update_calibration=True)
chi_exp.plot(analysis_e)

f_e_hz = float(analysis_e.metrics.get("f0", np.nan))
print(f"Resonator frequency (|e⟩): {f_e_hz / 1e9:.6f} GHz")

## 5. Extract Dispersive Shift χ

χ = f_resonator(|g⟩) − f_resonator(|e⟩)

In [ ]:
chi_hz = f_g_hz - f_e_hz
chi_mhz = chi_hz / 1e6

print(f"Dispersive shift χ = {chi_mhz:.6f} MHz")
print(f"                   = {chi_hz:.1f} Hz")
print(f"  f_g = {f_g_hz / 1e9:.6f} GHz")
print(f"  f_e = {f_e_hz / 1e9:.6f} GHz")

# Overlay plot
fig, ax = plt.subplots(figsize=(10, 6))

frequencies_g = np.asarray(analysis_g.data.get("frequencies", []), dtype=float)
S_g = np.asarray(analysis_g.data.get("S", []))
frequencies_e = np.asarray(analysis_e.data.get("frequencies", []), dtype=float)
S_e = np.asarray(analysis_e.data.get("S", []))

if frequencies_g.size > 0 and frequencies_e.size > 0:
    ax.plot(frequencies_g / 1e6, np.abs(S_g), label="|g⟩", color="#4c78a8", lw=1.5)
    ax.plot(frequencies_e / 1e6, np.abs(S_e), label="|e⟩", color="#e45756", lw=1.5)
    if np.isfinite(f_g_hz):
        ax.axvline(f_g_hz / 1e6, color="#4c78a8", ls="--", lw=1.2, alpha=0.7)
    if np.isfinite(f_e_hz):
        ax.axvline(f_e_hz / 1e6, color="#e45756", ls="--", lw=1.2, alpha=0.7)
    ax.set_xlabel("Frequency (MHz)")
    ax.set_ylabel("|S|")
    ax.set_title(f"Dispersive Shift: χ = {chi_mhz:.4f} MHz")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Sweep data not available for overlay plot.")

## 6. Apply Calibration & Save Checkpoint

In [ ]:
chi_patch, chi_patch_preview, chi_apply_result = preview_or_apply_patch_ops(
    session,
    reason="Dispersive shift χ measurement",
    proposed_patch_ops=analysis_e.metadata.get("proposed_patch_ops", []),
    apply=APPLY_CHI_CALIBRATION,
)

if chi_apply_result is not None:
    context_snapshot = getattr(session, "context_snapshot", None)
    attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
    if attr is None:
        raise RuntimeError("Calibration applied, but the refreshed cQED attribute snapshot is unavailable.")

stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="13_dispersive_shift_measurement",
    status="calibrated" if chi_apply_result is not None else "characterized",
    summary=f"Measured dispersive shift χ = {chi_mhz:.4f} MHz.",
    consumed_inputs={
        "freq_start_hz": CHI_FREQ_START,
        "freq_end_hz": CHI_FREQ_END,
        "df_hz": CHI_DF,
        "n_avg": CHI_N_AVG,
    },
    persisted_outputs={
        "applied": chi_apply_result is not None,
        "chi_hz": chi_hz,
    },
    advisory_outputs={
        "f_g_hz": f_g_hz,
        "f_e_hz": f_e_hz,
        "chi_mhz": chi_mhz,
    },
    next_stage="14_gate_calibration_benchmarking",
    notes=[
        "χ is used for readout modeling and dispersive readout optimization.",
        "Downstream notebooks may use this for number-splitting frequency calculations.",
    ],
    metrics={
        "chi_hz": chi_hz,
        "chi_mhz": chi_mhz,
        "f_g_hz": f_g_hz,
        "f_e_hz": f_e_hz,
    },
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")